# Inhaltsanalyse_Objekterkennung 

### Ziel
Analyse des **visuellen Inhalts der Frames**, um zu bestimmen, **was im Video zu sehen ist**.  
Dabei sollen bestimmte Objekte (z. B. Personen, Tiere) erkannt und quantifiziert werden.

---

## Zu erstellende Features

| Feature | Beschreibung | Wertebereich |
|----------|---------------|---------------|
| **ist_person_prominent** | Ist in (fast) jedem Frame eine Person sichtbar? | 1 = Ja / 0 = Nein |
| **ist_tier_sichtbar** | Wird mindestens einmal ein Tier (Hund oder Katze) erkannt? | 1 = Ja / 0 = Nein |
| **avg_objekte_pro_frame** | Durchschnittliche Anzahl erkannter Objekte pro Frame | numerisch |

---

## Algorithmus

Zur Objekterkennung wird das **YOLO-Modell (You Only Look Once)** verwendet.  
Empfohlen ist **YOLOv8n (nano)** von *Ultralytics*, da es:

- sehr **schnell** und **leichtgewichtig** ist,  
- und auf den **80 Standard-COCO-Klassen** trainiert wurde  
  (z. B. *Person, Hund, Katze, Auto, Tasse,* etc.).

---

## Prozessschritte

1. **Lade bestehende Features**
   - Öffne die Datei `features/video_features.csv`.

2. **Lade das YOLO-Modell**
   ```python
   from ultralytics import YOLO
   model = YOLO('yolov8n.pt')
   ```
3.	Iteriere durch jedes Video
	-	Für jede video_id in der CSV:
	-	Wähle 3–5 repräsentative Frames (z. B. zufällig oder gleichmäßig verteilt).
	-	Führe das YOLO-Modell auf diesen Frames aus.
4.	Analysiere die Erkennungen
	-	Zähle alle erkannten Objekte pro Frame.
	-	Prüfe, ob Personen regelmäßig erkannt werden.
	-	Prüfe, ob Tiere (Hund, Katze) mindestens einmal vorkommen.
	-	Berechne den Durchschnitt der erkannten Objekte pro Frame.
5.	Speichere neue Features
	-	Ergänze den DataFrame um die neuen Spalten:
	-	ist_person_prominent
	-	ist_tier_sichtbar
	-	avg_objekte_pro_frame
	-	Speichere die aktualisierte CSV:

## 1. Installation & Setup

In [ ]:
import os
import pandas as pd
import glob
from ultralytics import YOLO
import numpy as np

# --- Konfiguration ---
# 1. Eingabe: Die in T3_02 erstellte CSV
FEATURE_FILE = "features/video_features.csv"

# 2. Eingabe: Der Ordner mit den Frames
FRAME_DIR = "data/processed/video_frames"

# 3. Analyse-Parameter
# Wie viele Frames pro Video analysieren? (Mehr = genauer, aber langsamer)
FRAMES_TO_ANALYZE_PER_VIDEO = 5

# Klassen, nach denen wir suchen (aus dem COCO-Datensatz)
# 0: 'person', 16: 'dog', 15: 'cat'
INTERESTING_CLASSES = {
    'person': 0,
    'dog': 16,
    'cat': 15   
}

# Beurteilung des Prozesses (Fehlersuche)

Diese Phase war die mit Abstand problematischste des gesamten Analyse-Strangs. Die anfängliche Implementierung (mit `detector_backend='opencv'`) schlug fehl und lieferte für alle 197 Videos `dominante_emotion = 'none'`.

---

## Analyse des Problems

Die Analyse im Test Debugging-Notebook offenbarte das Kernproblem:  
Ein massiver Versionskonflikt zwischen **tensorflow**, **numpy** und **python**.

---

## Erkenntnis (Dependency Hell)

- **Python 3.13** war zu neu für `tensorflow==2.11.0` (benötigt von deepface).
- Die installierte **numpy==2.x** war inkompatibel mit `tensorflow==2.11.0`.
- Die installierte **opencv-python==4.12** war inkompatibel mit `numpy==1.x`.

---

## Lösung (Siehe Projektdokumentation)

Das **„Einfrieren“ des gesamten Environments** auf folgende Versionen war die einzige stabile Lösung:

| Komponente | Version |
|-------------|----------|
| Python | 3.10 |
| TensorFlow | 2.11 |
| Numpy | 1.24 |
| Keras | 2.11 |
| OpenCV | 4.7 |
| MTCNN | 0.1.0 |

## 2. Laden von Modell und Daten
Laden des (kleine, schnelle) YOLOv8n-Modells und der CSV-Datei.

In [7]:
# Lade das Modell. Beim ersten Mal wird es automatisch heruntergeladen.
try:
    model = YOLO('yolov8n.pt') # 'n' = nano. Schnellstes Modell.
    print("YOLOv8n-Modell erfolgreich geladen.")
except Exception as e:
    print(f"FEHLER beim Laden des YOLO-Modells: {e}")
    print("Stelle sicher, dass 'ultralytics' installiert ist und eine Internetverbindung besteht.")

YOLOv8n-Modell erfolgreich geladen.


In [ ]:
# Lade die CSV-Datei
try:
    df = pd.read_csv(FEATURE_FILE)
    if df.empty:
        print(f"WARNUNG: {FEATURE_FILE} ist leer.")
        print("Stelle sicher, dass T3_02 (Dynamic Analysis) erfolgreich durchgelaufen ist.")
    else:
        print(f"{len(df)} Videos aus {FEATURE_FILE} geladen.")
        
except FileNotFoundError:
    print(f"FEHLER: {FEATURE_FILE} nicht gefunden!")
    print("Stelle sicher, dass T3_02 (Dynamic Analysis) erfolgreich durchgelaufen ist.")
    raise # Stoppt die Ausführung des Notebooks
except Exception as e:
    print(f"FEHLER beim Laden von {FEATURE_FILE}: {e}")
    raise

# Neue Spalten initialisieren 
if 'ist_person_prominent' not in df.columns:
    df['ist_person_prominent'] = 0
if 'ist_tier_sichtbar' not in df.columns:
    df['ist_tier_sichtbar'] = 0
if 'avg_objekte_pro_frame' not in df.columns:
    df['avg_objekte_pro_frame'] = 0.0

df.head()

197 Videos aus features/video_features.csv geladen.


,video_id,schnitt_frequenz,durchschnittliche_bewegung,anzahl_frames,video_dauer_sek,ist_person_prominent,ist_tier_sichtbar,avg_objekte_pro_frame
0,top_53_likes_729700_id_7542648831586880823,0.214953,9.150545,107,107.0,0,0,0.0
1,top_45_likes_780200_id_7548598130665622804,0.192308,6.863052,26,26.0,0,0,0.0
2,top_96_likes_365300_id_7560113050100043026,0.290909,5.528583,55,55.0,0,0,0.0
3,top_32_likes_1000000_id_7556001068405050638,0.110672,9.759456,253,253.0,0,0,0.0
4,top_19_likes_1500000_id_7231352152743152942,0.000000,3.906820,29,29.0,0,0,0.0


## 3. Hauptverarbeitung: Objekterkennung
Iteration durch den DataFrame und Analyse der Frames für jedes Video.

Wichtiger Hinweis: Diese Zelle wird lange laufen!.

In [ ]:
print("Starte Objekterkennung für alle Videos...")

# Iteriere durch jede Zeile (jedes Video) im DataFrame
# 'index' wird benötigt, um den Wert später zurück in den DataFrame zu schreiben
for index, row in df.iterrows():
    video_id = row['video_id']
    
    # 3a. Finde die Frames für dieses Video
    frame_files_pattern = os.path.join(FRAME_DIR, f"{video_id}_frame_*.jpg")
    video_frames = glob.glob(frame_files_pattern)
    
    if not video_frames:
        print(f"WARNUNG: Keine Frames für {video_id} gefunden. Überspringe.")
        continue
        
    # 3b. Wähle Frames für die Analyse aus (Sampling)
    # Wähle gleichmäßig verteilte Frames aus der Liste
    if len(video_frames) > FRAMES_TO_ANALYZE_PER_VIDEO:
        indices = np.linspace(0, len(video_frames) - 1, FRAMES_TO_ANALYZE_PER_VIDEO, dtype=int)
        frames_to_process = [video_frames[i] for i in indices]
    else:
        frames_to_process = video_frames # Nimm alle, wenn es weniger als 5 sind
        
    # print(f"Analysiere {len(frames_to_process)} Frames für {video_id}...")
    
    total_objects = 0
    person_detections = 0
    animal_detections = 0
    
    # 3c. Führe YOLO auf den ausgewählten Frames aus
    try:
        results = model.predict(frames_to_process, verbose=False) # verbose=False verhindert Log-Spam
    except Exception as e:
        print(f"FEHLER bei YOLO.predict für {video_id}: {e}")
        continue
        
    # 3d. Analysiere die Ergebnisse
    for r in results:
        # r.boxes.cls ist eine Liste der erkannten Klassen-IDs (z.B. [0, 0, 16])
        detected_classes = r.boxes.cls.cpu().numpy()
        
        total_objects += len(detected_classes)
        
        if INTERESTING_CLASSES['person'] in detected_classes:
            person_detections += 1
            
        if INTERESTING_CLASSES['dog'] in detected_classes or INTERESTING_CLASSES['cat'] in detected_classes:
            animal_detections += 1

    # 3e. Berechne die finalen Features
    num_analyzed = len(frames_to_process)
    
    # Ist Person "prominent"? (d.h. in >80% der analysierten Frames) - überwiegend sichtbar
    ist_person_prominent = 1 if (person_detections / num_analyzed) >= 0.8 else 0
    
    # Ist "Tier" sichtbar? (d.h. in mindestens einem Frame)
    ist_tier_sichtbar = 1 if animal_detections > 0 else 0
    
    # Durchschnittliche Objektanzahl
    avg_objekte_pro_frame = total_objects / num_analyzed if num_analyzed > 0 else 0
    
    # 3f. Speichere Features zurück in den DataFrame
    df.loc[index, 'ist_person_prominent'] = ist_person_prominent
    df.loc[index, 'ist_tier_sichtbar'] = ist_tier_sichtbar
    df.loc[index, 'avg_objekte_pro_frame'] = avg_objekte_pro_frame

    # Log-Ausgabe alle 20 Videos
    if (index + 1) % 20 == 0:
        print(f"Fortschritt: {index + 1} / {len(df)} Videos verarbeitet.")

print("\n--- Objekterkennung abgeschlossen ---")

Starte Objekterkennung für alle Videos...
Fortschritt: 20 / 197 Videos verarbeitet.
Fortschritt: 40 / 197 Videos verarbeitet.
Fortschritt: 60 / 197 Videos verarbeitet.
Fortschritt: 80 / 197 Videos verarbeitet.
Fortschritt: 100 / 197 Videos verarbeitet.
Fortschritt: 120 / 197 Videos verarbeitet.
Fortschritt: 140 / 197 Videos verarbeitet.
Fortschritt: 160 / 197 Videos verarbeitet.
Fortschritt: 180 / 197 Videos verarbeitet.

--- Objekterkennung abgeschlossen ---


## 4. Ergebnis speichern
Die video_features.csv wird nun mit den neuen Spalten überschrieben.

In [10]:
# 4. Ergebnisse in dieselbe CSV-Datei zurückspeichern
try:
    df.to_csv(FEATURE_FILE, index=False)
    print(f"Erfolgreich aktualisiert: {FEATURE_FILE}")
    
    # Zeige die ersten 5 Zeilen der aktualisierten Datei
    print("\nAktualisierte Datei-Vorschau (video_features.csv):")
    print(df[['video_id', 'schnitt_frequenz', 'ist_person_prominent', 'ist_tier_sichtbar', 'avg_objekte_pro_frame']].head())

except PermissionError:
    print(f"\nFEHLER: Keine Berechtigung, {FEATURE_FILE} zu schreiben.")
    print("Ist die Datei vielleicht in Excel oder einem anderen Programm geöffnet?")
except Exception as e:
    print(f"\nEin Fehler ist beim Speichern aufgetreten: {e}")

Erfolgreich aktualisiert: features/video_features.csv

Aktualisierte Datei-Vorschau (video_features.csv):
                                      video_id  schnitt_frequenz  \
0   top_53_likes_729700_id_7542648831586880823          0.214953   
1   top_45_likes_780200_id_7548598130665622804          0.192308   
2   top_96_likes_365300_id_7560113050100043026          0.290909   
3  top_32_likes_1000000_id_7556001068405050638          0.110672   
4  top_19_likes_1500000_id_7231352152743152942          0.000000   

   ist_person_prominent  ist_tier_sichtbar  avg_objekte_pro_frame  
0                     1                  0                    1.8  
1                     1                  0                    1.8  
2                     1                  0                    1.6  
3                     1                  0                    1.0  
4                     1                  0                    2.0  


# Beurteilung der finalen Emotions-Features

Die `video_features.csv` enthält nun die berechneten **Emotionsdaten**.

---

## Erkenntnis

Das Feature `dominante_emotion = 'none'` ist jetzt **ein valides Ergebnis**.  
Es bedeutet **nicht** „Fehler“, sondern:

> „In diesem Video wurde kein Gesicht gefunden.“

Diese Information ist **modellrelevant**, z. B. bei **Landschafts-, Produkt- oder Kochvideos**,  
wo keine Personen vorkommen.

---

## Stabilität

Nach der Environment-Reparatur erwies sich die **MTCNN-Bibliothek** als  
**sehr robust und zuverlässig** bei der Gesichtserkennung.